<a href="https://colab.research.google.com/github/yasuhisasugiura-crypto/newfile_generator/blob/main/prep_title_pos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 【T1-0】Drive をつなぎ、配布チャンクだけをローカルに戻す（GPU 不要）
# キャッシュの tar から、チャンクと chunk_offsets.npy だけを取り出す。インデックスやモデルは戻さない
from google.colab import drive
drive.mount("/content/drive")
import os, subprocess, time
CACHE = "/content/drive/MyDrive/LLM講座最終コンペ_cache"
CHUNKS_DIR = "/content/rag_data/data/chunks"
OFFSETS = "/content/rag_data/index/chunk_offsets.npy"
t0 = time.time()
if os.path.isdir(CHUNKS_DIR) and any(f.startswith("chunks-") for f in os.listdir(CHUNKS_DIR)) and os.path.isfile(OFFSETS):
    print("チャンクは復元済みなので飛ばす")
elif os.path.exists(CACHE + "/rag_data.tar"):
    print("tar から取り出す（tar 全体を読むので数分かかる）")
    subprocess.run(["tar", "-xf", CACHE + "/rag_data.tar", "-C", "/content",
                    "rag_data/data/chunks", "rag_data/index/chunk_offsets.npy"], check=True)
elif os.path.isdir(CACHE + "/rag_data/data/chunks"):
    print("フォルダから rsync で戻す")
    os.makedirs(CHUNKS_DIR, exist_ok=True)
    os.makedirs(os.path.dirname(OFFSETS), exist_ok=True)
    subprocess.run(["rsync", "-a", CACHE + "/rag_data/data/chunks/", CHUNKS_DIR + "/"], check=True)
    subprocess.run(["rsync", "-a", CACHE + "/rag_data/index/chunk_offsets.npy", OFFSETS], check=True)
else:
    raise SystemExit("キャッシュが見つからない: " + CACHE)
files = sorted(f for f in os.listdir(CHUNKS_DIR) if f.startswith("chunks-") and f.endswith(".jsonl"))
print("チャンクのファイル: %d 個 / %.0f 秒" % (len(files), time.time() - t0))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
チャンクは復元済みなので飛ばす
チャンクのファイル: 491 個 / 0 秒


In [ ]:
# 【T1-1】配布チャンクから「記事名 → chunk_id の一覧」を作り、Drive に新しい名前で保存する
# チャンクの本文は保存しない。既存のファイルがあれば止める（上書きしない）
import json, os, time, hashlib, collections
import numpy as np
OUT_DIR = "/content/drive/MyDrive/LLM講座最終コンペ_eval/title_index"
OUT = OUT_DIR + "/title_to_chunks.json"
MAN = OUT_DIR + "/title_index_manifest.json"
for p in (OUT, MAN):
    assert not os.path.exists(p), "すでにある（上書き防止）: " + p
os.makedirs(OUT_DIR, exist_ok=True)

t0 = time.time()
files = sorted(f for f in os.listdir(CHUNKS_DIR) if f.startswith("chunks-") and f.endswith(".jsonl"))
title2ids = {}
n, not_seq, keys0 = 0, 0, None
for fn in files:
    with open(os.path.join(CHUNKS_DIR, fn), encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            c = json.loads(line)
            if keys0 is None:
                keys0 = sorted(c.keys())
            cid = int(c["chunk_id"])
            if cid != n:
                not_seq += 1
            title2ids.setdefault(c["title"], []).append(cid)
            n += 1
print("読み込み: %d チャンク / %.0f 秒" % (n, time.time() - t0))
print("1件目のキー:", keys0)
print("chunk_offsets.npy の件数:", len(np.load(OFFSETS)), "/ chunk_id が通し番号と違うチャンク:", not_seq)

# --- 集計（中身は出さない）---
lens = np.array([len(v) for v in title2ids.values()])
tlen = collections.Counter(min(len(t), 6) for t in title2ids)
non_contig = sum(1 for v in title2ids.values() if v != list(range(v[0], v[0] + len(v))))
print("記事名の数:", len(title2ids), "（方針の想定: 49,043）")
print("1記事あたりのチャンク数: 中央値 %d / 90%% 点 %d / 99%% 点 %d / 最大 %d"
      % (np.median(lens), np.percentile(lens, 90), np.percentile(lens, 99), lens.max()))
print("記事名の文字数（6 は6文字以上）:", dict(sorted(tlen.items())))
print("括弧を含む記事名:", sum(1 for t in title2ids if "(" in t or "（" in t))
print("chunk_id が連続していない記事:", non_contig)

# --- 保存 ---
tmp = OUT + ".tmp"
with open(tmp, "w", encoding="utf-8") as f:
    json.dump(title2ids, f, ensure_ascii=False)
os.replace(tmp, OUT)
sha = hashlib.sha256(open(OUT, "rb").read()).hexdigest()
json.dump({"file": os.path.basename(OUT), "sha256": sha, "n_chunks": n, "n_titles": len(title2ids),
           "source": "配布チャンク chunks-*.jsonl の title と chunk_id", "created_at_utc": time.strftime("%Y-%m-%d %H:%M", time.gmtime())},
          open(MAN, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print("保存した: %s（%.1f MB）/ sha256 先頭 %s" % (OUT.replace("/content/drive/MyDrive/", ""), os.path.getsize(OUT) / 1e6, sha[:12]))

AssertionError: すでにある（上書き防止）: /content/drive/MyDrive/LLM講座最終コンペ_eval/title_index/title_to_chunks.json

In [ ]:
# 【T1-2】開発用の質問（320問）に、記事名がいくつ含まれるかを数える（質問文・記事名は表示しない）
import json, unicodedata, collections
QP = "/content/drive/MyDrive/LLM講座最終コンペ_eval/dev_work/questions.json"
qs = json.load(open(QP, encoding="utf-8"))
assert isinstance(qs, list) and len(qs) == 320 and all(str(q["q_id"]).startswith("dev") for q in qs), "開発用の質問ではないので止める"
nf = lambda s: unicodedata.normalize("NFKC", s)
titles = [(nf(t), len(nf(t))) for t in title2ids]
match_lens = []   # 問題ごとに、含まれる記事名の文字数のリスト
for q in qs:
    qt = nf(q["question"])
    match_lens.append([L for t, L in titles if L >= 2 and t in qt])
for minlen in (2, 3, 4, 5):
    cnt = [sum(1 for L in m if L >= minlen) for m in match_lens]
    has = sum(1 for c in cnt if c > 0)
    print("記事名が %d文字以上: 1つ以上含む問題 %3d / 320 ・1問あたりの数 中央値 %d・最大 %d"
          % (minlen, has, sorted(cnt)[len(cnt) // 2], max(cnt)))
longest = collections.Counter(min(max(m), 10) if m else 0 for m in match_lens)
print("問題ごとの一番長い記事名の文字数（0 は該当なし・10 は10文字以上）:", dict(sorted(longest.items())))

記事名が 2文字以上: 1つ以上含む問題 139 / 320 ・1問あたりの数 中央値 0・最大 6
記事名が 3文字以上: 1つ以上含む問題 108 / 320 ・1問あたりの数 中央値 0・最大 6
記事名が 4文字以上: 1つ以上含む問題  87 / 320 ・1問あたりの数 中央値 0・最大 6
記事名が 5文字以上: 1つ以上含む問題  52 / 320 ・1問あたりの数 中央値 0・最大 6
問題ごとの一番長い記事名の文字数（0 は該当なし・10 は10文字以上）: {0: 181, 2: 31, 3: 21, 4: 35, 5: 14, 6: 12, 7: 10, 8: 4, 9: 1, 10: 11}


In [ ]:
# 【T2-1】学習データ（1500問）で、正解を含む資料が何番目にあるかを数える（質問文・正解は表示しない）
# 判定は LoRA-B の B' セルと同じ（NFKC・小文字・空白除去のあとで、正解の文字列が「タイトル＋本文」に含まれるか）
import json, re, unicodedata, collections
D = "/content/drive/MyDrive/LLM講座最終コンペ_lora"
rows = [json.loads(l) for l in open(D + "/train_contexts_1500.jsonl", encoding="utf-8") if l.strip()]
ans = json.load(open(D + "/train_answers.json", encoding="utf-8"))
assert len(rows) == 1500 and all(r["q_id"].startswith("tr") for r in rows), "学習データ（tr・1500問）ではないので止める"
_norm = lambda s: re.sub(r"\s+", "", unicodedata.normalize("NFKC", s or "").lower())
YESNO = {"はい", "いいえ", "yes", "no"}

first, anyhit, nhits, first_short = collections.Counter(), collections.Counter(), collections.Counter(), collections.Counter()
n_yn = n_none = 0
for r in rows:
    a = ans[r["q_id"]]["answers"]
    if a and all(_norm(x) in YESNO for x in a):
        n_yn += 1
        continue
    hits = [i + 1 for i, c in enumerate(r["contexts"][:5])
            if any(_norm(x) and _norm(x) in _norm(c["title"] + c["text"]) for x in a)]
    if not hits:
        n_none += 1
        continue
    first[hits[0]] += 1
    for h in hits:
        anyhit[h] += 1
    nhits[len(hits)] += 1
    if max(len(_norm(x)) for x in a) <= 2:
        first_short[hits[0]] += 1

n_hit = sum(first.values())
print("はい/いいえ: %d / 資料5件に正解なし: %d / 正解あり: %d（学習に使える見込み %d。B' の記録: 1038〜1039）"
      % (n_yn, n_none, n_hit, n_yn + n_hit))
print("正解を含む資料のうち、一番上にあるものの位置:")
for k in range(1, 6):
    print("  %d番目: %4d（%4.1f%%）/ うち正解が2文字以下 %d" % (k, first[k], 100 * first[k] / max(1, n_hit), first_short[k]))
p1 = first[1] / max(1, n_hit)
p345 = (first[3] + first[4] + first[5]) / max(1, n_hit)
print("正解を含む資料が全部で何件あるか:", dict(sorted(nhits.items())))
print("位置ごとの「正解を含む資料」の延べ数:", dict(sorted(anyhit.items())))
print("判定: 1番目 %.1f%% / 3〜5番目 %.1f%% →" % (100 * p1, 100 * p345),
      "LoRA-D を進める" if (p1 >= 0.6 and p345 <= 0.2) else "基準に当てはまらない（結果を貼って相談）")

はい/いいえ: 290 / 資料5件に正解なし: 461 / 正解あり: 749（学習に使える見込み 1039。B' の記録: 1038〜1039）
正解を含む資料のうち、一番上にあるものの位置:
  1番目:  558（74.5%）/ うち正解が2文字以下 70
  2番目:   98（13.1%）/ うち正解が2文字以下 11
  3番目:   46（ 6.1%）/ うち正解が2文字以下 12
  4番目:   28（ 3.7%）/ うち正解が2文字以下 4
  5番目:   19（ 2.5%）/ うち正解が2文字以下 2
正解を含む資料が全部で何件あるか: {1: 179, 2: 130, 3: 114, 4: 129, 5: 197}
位置ごとの「正解を含む資料」の延べ数: {1: 558, 2: 489, 3: 461, 4: 409, 5: 365}
判定: 1番目 74.5% / 3〜5番目 12.4% → LoRA-D を進める


In [1]:
# [B0-0] JEMHopQA の derivations から橋渡しの記事名 Y を取り出し、使えるか調べる（CPU・学習しない・本番データは使わない）
from google.colab import drive
drive.mount("/content/drive")
import os, json, glob, subprocess, unicodedata, re, collections

EVAL = "/content/drive/MyDrive/LLM講座最終コンペ_eval"
OUTD = EVAL + "/bridge"
REPO_DIR = "/content/JEMHopQA"
assert "LLM講座最終コンペ_work" not in OUTD

def nz(s):
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(s or "")).lower())

# ---- 1) JEMHopQA を取得（使用可能データ一覧の取得元） ----
if not os.path.isdir(REPO_DIR):
    r = subprocess.run(["git", "clone", "--depth", "1", "https://github.com/aiishii/JEMHopQA.git", REPO_DIR],
                       capture_output=True, text=True)
    print("git clone 終了コード:", r.returncode)
    if r.returncode != 0:
        print(r.stderr[-800:])
        raise SystemExit("JEMHopQA の取得に失敗した。上の表示を貼る")
files = sorted(glob.glob(REPO_DIR + "/**/*.json", recursive=True))
print("json ファイル:", [f.replace(REPO_DIR + "/", "") for f in files])

def pick(kind):
    c = [f for f in files if os.path.basename(f).startswith(kind) and "1.2" in f]
    if len(c) != 1:
        raise SystemExit("%s の ver1.2 のファイルが1つに決まらない: %s。上の一覧を貼る" % (kind, c))
    return c[0]

def load(p):
    d = json.load(open(p, encoding="utf-8"))
    return d if isinstance(d, list) else list(d.values())

TRAIN_P, DEV_P = pick("train"), pick("dev")
train, dev = load(TRAIN_P), load(DEV_P)
print("train: %s %d 問 / dev: %s %d 問" % (os.path.basename(TRAIN_P), len(train), os.path.basename(DEV_P), len(dev)))
print("1件目のキー:", sorted(train[0].keys()))
print("1件目の type:", train[0].get("type"), "/ derivations:", train[0].get("derivations"))

# ---- 2) Y を取り出す規則（構成型・d1 の目的語 = d2 の主語） ----
def bridge(rec):
    if rec.get("type") != "compositional":
        return None, "構成型ではない"
    d = rec.get("derivations") or []
    if len(d) < 2:
        return None, "三つ組が2つ未満"
    try:
        o = d[0][2]
        Y = o[0] if isinstance(o, list) else o
        s2 = d[1][0]
    except Exception:
        return None, "形が想定外"
    if not str(Y).strip():
        return None, "Y が空"
    if nz(Y) != nz(s2):
        return None, "d1の目的語 ≠ d2の主語"
    return str(Y), "OK"

# ---- 3) 配布チャンクの記事名 ----
T = json.load(open(EVAL + "/title_index/title_to_chunks.json", encoding="utf-8"))
TK = {nz(k) for k in T}
TK2 = TK | {nz(re.sub(r"\s*[（(][^）)]*[）)]\s*$", "", k)) for k in T}   # 括弧を外した名前でも引く
def in_title(Y):
    return "記事名と一致" if nz(Y) in TK else ("括弧を外すと一致" if nz(Y) in TK2 else "なし")
print("\n記事名の数:", len(T))

# ---- 4) train ----
print("\n== train ==")
print("type の内訳:", dict(collections.Counter(r.get("type") for r in train)))
st, cov, tr_rows = collections.Counter(), collections.Counter(), []
for r in train:
    Y, s = bridge(r)
    st[s] += 1
    if Y:
        c = in_title(Y); cov[c] += 1
        tr_rows.append({"qid": r.get("qid", r.get("id")), "question": r.get("question"), "Y": Y, "title": c})
print("Y の取り出し:", dict(st))
print("Y が記事名にあるか:", dict(cov))
n_ok = len(tr_rows); n_cov = cov["記事名と一致"] + cov["括弧を外すと一致"]
print("使える Y: %d 問 / うち記事名にある %d 問（%.1f%%）" % (n_ok, n_cov, 100 * n_cov / max(n_ok, 1)))
print("例（Y だけ）:", [x["Y"] for x in tr_rows[:5]])

# ---- 5) 開発用セット（JEMHopQA の 120 問）----
print("\n== 開発用セット ==")
DQ = json.load(open(EVAL + "/dev_work/questions.json", encoding="utf-8"))
G = json.load(open(EVAL + "/dev_data/dev_answers.json", encoding="utf-8"))
G = G if isinstance(G, dict) else {r["q_id"]: r for r in G}
CTX = {}
for l in open(EVAL + "/dev_contexts.jsonl", encoding="utf-8"):
    if l.strip():
        o = json.loads(l); CTX[o["q_id"]] = o["contexts"]
by_text = {nz(r.get("question")): r for r in dev}
assert len(DQ) == 320 and all(str(q["q_id"]).startswith("dev") for q in DQ), "開発用の質問ではない"

def is_yn(a):
    return bool(a) and all(str(x).strip().upper() in ("YES", "NO") for x in a)
def hit(ctxs, ans):
    gs = [nz(a) for a in ans if nz(a)]
    return any(g in nz(c["title"] + c["text"]) for c in ctxs for g in gs)

cnt, dv_rows = collections.Counter(), []
target = []
for q in DQ:
    qid = q["q_id"]
    if "jem" not in str(G[qid].get("source", "")).lower():
        continue
    cnt["jemhopqa"] += 1
    rec = by_text.get(nz(q["question"]))
    if rec is None:
        cnt["JEMHopQA dev と質問文が一致しない"] += 1; continue
    Y, s = bridge(rec)
    ans = G[qid]["answers"]
    yn = is_yn(ans)
    cnt["%s・%s" % ("はい/いいえ" if yn else "非yn", "構成型" if rec.get("type") == "compositional" else "比較型")] += 1
    if Y:
        dv_rows.append({"q_id": qid, "Y": Y, "title": in_title(Y), "yn": yn})
    if yn or not Y:
        continue
    h5, h10 = hit(CTX[qid][:5], ans), hit(CTX[qid][:10], ans)
    cnt["非yn構成型・Y あり"] += 1
    cnt["  上位5件に正解あり"] += h5
    cnt["  上位10件に正解なし"] += (not h10)
    if not h5:
        cnt["  上位5件に正解なし"] += 1
        if in_title(Y) != "なし":
            cnt["  └ そのうち Y が記事名にある（Branch B の対象）"] += 1
            target.append(qid)
for k, v in cnt.items():
    print("  %-44s %d" % (k, v))
print("Branch B の対象の q_id:", target)

# ---- 6) E3b の「正しい Y」（私の知識）と derivations の Y の照合 ----
E3B = {"dev0031": "マークII", "dev0041": "マークX", "dev0058": "生田竜聖", "dev0071": "岡本太郎",
       "dev0108": "藤本タツキ", "dev0118": "徳川家光", "dev0129": "トヨタ自動車", "dev0185": "愛子",
       "dev0186": "大人計画", "dev0210": "Apple", "dev0211": "ワイルド・スピード ICE BREAK",
       "dev0224": "アップル・インク", "dev0258": "日本プロ野球名球会", "dev0261": "徳川家光", "dev0263": "許斐剛"}
dY = {r["q_id"]: r["Y"] for r in dv_rows}
print("\n== E3b の Y との照合 ==")
same = 0
for k, v in E3B.items():
    y = dY.get(k)
    ok = y is not None and nz(y) == nz(v)
    same += ok
    print("  %s  E3b: %-24s derivations: %-24s %s" % (k, v, y, "一致" if ok else "違う"))
print("一致 %d / %d" % (same, len(E3B)))

# ---- 7) 保存（すでにあれば作らない） ----
os.makedirs(OUTD, exist_ok=True)
for name, rows in (("jemhop_train_bridge.jsonl", tr_rows), ("jemhop_dev_bridge.jsonl", dv_rows)):
    p = OUTD + "/" + name
    if os.path.exists(p):
        print("すでにあるので作らない:", p); continue
    with open(p, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print("保存:", p, len(rows), "行")

# ---- 8) 判定 ----
okA = n_ok >= 300 and n_cov >= 0.7 * n_ok
okB = len(target) >= 10
print("\n判定A（train：使える Y 300 問以上・記事名にある割合 70%% 以上）: %s" % ("OK" if okA else "NG"))
print("判定B（開発用：Branch B の対象 10 問以上）: %s（%d 問）" % ("OK" if okB else "NG", len(target)))
print("→ B0-1（正しい Y で再検索する上限の測定）に進む" if okA and okB else "→ 結果を貼る（Branch B の扱いを相談する）")

Mounted at /content/drive
git clone 終了コード: 0
json ファイル: ['corpus/dev.json', 'corpus/dev_for_view.json', 'corpus/train.json', 'corpus/train_for_view.json', 'corpus_ver1.1/dev_for_view_ver1.1.json', 'corpus_ver1.1/dev_ver1.1.json', 'corpus_ver1.1/train_for_view_ver1.1.json', 'corpus_ver1.1/train_ver1.1.json', 'corpus_ver1.2/dev_ver1.2.json', 'corpus_ver1.2/train_ver1.2.json']
train: train_ver1.2.json 1059 問 / dev: dev_ver1.2.json 120 問
1件目のキー: ['answer', 'derivations', 'page_ids', 'qid', 'question', 'time_dependent', 'type']
1件目の type: comparison / derivations: [['仮面ライダー電王', '放送回数', ['49']], ['あまちゃん', '放送回数', ['156']]]

記事名の数: 49043

== train ==
type の内訳: {'comparison': 667, 'compositional': 392}
Y の取り出し: {'構成型ではない': 667, 'OK': 369, 'd1の目的語 ≠ d2の主語': 23}
Y が記事名にあるか: {'なし': 151, '記事名と一致': 218}
使える Y: 369 問 / うち記事名にある 218 問（59.1%）
例（Y だけ）: ['平安京', '仲代達矢', '吉本興業', '榮倉奈々', 'SMエンターテインメント']

== 開発用セット ==
  jemhopqa                                     120
  非yn・比較型                              